## Εισαγωγή 

Αυτή η ενότητα θα καλύψει: 
- Τι είναι η κλήση συνάρτησης και οι περιπτώσεις χρήσης της 
- Πώς να δημιουργήσετε μια κλήση συνάρτησης χρησιμοποιώντας το Azure OpenAI 
- Πώς να ενσωματώσετε μια κλήση συνάρτησης σε μια εφαρμογή 

## Στόχοι Μάθησης 

Μετά την ολοκλήρωση αυτής της ενότητας θα γνωρίζετε πώς και θα κατανοείτε: 

- Τον σκοπό χρήσης της κλήσης συνάρτησης 
- Διαμόρφωση Κλήσης Συνάρτησης χρησιμοποιώντας την Υπηρεσία Azure Open AI 
- Σχεδιασμό αποτελεσματικών κλήσεων συναρτήσεων για την περίπτωση χρήσης της εφαρμογής σας 


## Κατανόηση των Κλήσεων Συνάρτησης 

Για αυτό το μάθημα, θέλουμε να δημιουργήσουμε μια λειτουργία για το εκπαιδευτικό μας startup που επιτρέπει στους χρήστες να χρησιμοποιούν ένα chatbot για να βρουν τεχνικά μαθήματα. Θα προτείνουμε μαθήματα που ταιριάζουν στο επίπεδο δεξιοτήτων τους, στον τρέχοντα ρόλο και στην τεχνολογία ενδιαφέροντός τους. 

Για να ολοκληρώσουμε αυτό, θα χρησιμοποιήσουμε έναν συνδυασμό: 
 - `Azure Open AI` για να δημιουργήσουμε μια εμπειρία συνομιλίας για τον χρήστη
 - `Microsoft Learn Catalog API` για να βοηθήσουμε τους χρήστες να βρουν μαθήματα βάσει του αιτήματος του χρήστη 
 - `Function Calling` για να πάρουμε το ερώτημα του χρήστη και να το στείλουμε σε μια συνάρτηση για να κάνουμε το αίτημα API. 

Για να ξεκινήσουμε, ας δούμε γιατί θα θέλαμε να χρησιμοποιήσουμε την κλήση συνάρτησης στην πρώτη θέση: 


### Γιατί Κλήση Συνάρτησης

Αν έχετε ολοκληρώσει οποιοδήποτε άλλο μάθημα σε αυτό το μάθημα, πιθανότατα καταλαβαίνετε τη δύναμη της χρήσης Μεγάλων Γλωσσικών Μοντέλων (LLMs). Ελπίζουμε επίσης να μπορείτε να δείτε μερικούς από τους περιορισμούς τους.

Η Κλήση Συνάρτησης είναι μια δυνατότητα της Υπηρεσίας Azure Open AI για να ξεπεραστούν οι ακόλουθοι περιορισμοί:
1) Συνεπής μορφή απόκρισης
2) Ικανότητα χρήσης δεδομένων από άλλες πηγές μιας εφαρμογής σε ένα πλαίσιο συνομιλίας

Πριν από την κλήση συνάρτησης, οι απαντήσεις από ένα LLM ήταν άδομη και ασυνεπείς. Οι προγραμματιστές έπρεπε να γράφουν πολύπλοκο κώδικα επικύρωσης για να διασφαλίσουν ότι μπορούν να χειριστούν κάθε παραλλαγή απάντησης.

Οι χρήστες δεν μπορούσαν να λάβουν απαντήσεις όπως «Ποιος είναι ο τρέχων καιρός στη Στοκχόλμη;». Αυτό συμβαίνει επειδή τα μοντέλα περιορίζονταν στον χρόνο κατά τον οποίο εκπαιδεύτηκαν τα δεδομένα.

Ας δούμε το παρακάτω παράδειγμα που απεικονίζει αυτό το πρόβλημα:

Ας πούμε ότι θέλουμε να δημιουργήσουμε μια βάση δεδομένων με δεδομένα φοιτητών ώστε να μπορούμε να προτείνουμε το σωστό μάθημα σε αυτούς. Παρακάτω έχουμε δύο περιγραφές φοιτητών που είναι πολύ παρόμοιες στα δεδομένα που περιέχουν.


In [ ]:
student_1_description="Emily Johnson is a sophomore majoring in computer science at Duke University. She has a 3.7 GPA. Emily is an active member of the university's Chess Club and Debate Team. She hopes to pursue a career in software engineering after graduating."
 
student_2_description = "Michael Lee is a sophomore majoring in computer science at Stanford University. He has a 3.8 GPA. Michael is known for his programming skills and is an active member of the university's Robotics Club. He hopes to pursue a career in artificial intelligence after finishing his studies."


Θέλουμε να στείλουμε αυτό σε ένα LLM για να αναλύσει τα δεδομένα. Αυτό μπορεί αργότερα να χρησιμοποιηθεί στην εφαρμογή μας για να το στείλουμε σε ένα API ή να το αποθηκεύσουμε σε μια βάση δεδομένων.

Ας δημιουργήσουμε δύο πανομοιότυπα prompts που καθοδηγούμε το LLM για το ποιες πληροφορίες μας ενδιαφέρουν:


Θέλουμε να στείλουμε αυτό σε ένα LLM για να αναλύσει τα μέρη που είναι σημαντικά για το προϊόν μας. Έτσι μπορούμε να δημιουργήσουμε δύο ταυτόσημες προτροπές για να καθοδηγήσουμε το LLM: 


In [ ]:
prompt1 = f'''
Please extract the following information from the given text and return it as a JSON object:

name
major
school
grades
club

This is the body of text to extract the information from:
{student_1_description}
'''


prompt2 = f'''
Please extract the following information from the given text and return it as a JSON object:

name
major
school
grades
club

This is the body of text to extract the information from:
{student_2_description}
'''


Αφού δημιουργήσουμε αυτές τις δύο εντολές, θα τις στείλουμε στο LLM χρησιμοποιώντας το `client.responses.create`. Αποθηκεύουμε την εντολή στη μεταβλητή `input` και αναθέτουμε τον ρόλο σε `user`. Αυτό γίνεται για να μιμηθούμε ένα μήνυμα από έναν χρήστη που γράφεται σε ένα chatbot. 



In [ ]:
import os
import json
from openai import OpenAI
from dotenv import load_dotenv
load_dotenv()

# The OpenAI client points at the Azure OpenAI (Microsoft Foundry) /openai/v1/ endpoint
client = OpenAI(
  api_key=os.environ['AZURE_OPENAI_API_KEY'],
  base_url=f"{os.environ['AZURE_OPENAI_ENDPOINT'].rstrip('/')}/openai/v1/",
  )

deployment=os.environ['AZURE_OPENAI_DEPLOYMENT']


: 

Τώρα μπορούμε να στείλουμε και τα δύο αιτήματα στο LLM και να εξετάσουμε την απάντηση που λαμβάνουμε. 


In [ ]:
openai_response1 = client.responses.create(
 model=deployment,    
 input = [{'role': 'user', 'content': prompt1}],
 store=False,
)
openai_response1.output_text 


In [ ]:
openai_response2 = client.responses.create(
 model=deployment,    
 input = [{'role': 'user', 'content': prompt2}],
 store=False,
)
openai_response2.output_text


In [ ]:
# Loading the response as a JSON object
json_response1 = json.loads(openai_response1.output_text)
json_response1


In [ ]:
# Loading the response as a JSON object
json_response2 = json.loads(openai_response2.output_text)
json_response2



Παρόλο που τα προτροπές είναι τα ίδια και οι περιγραφές παρόμοιες, μπορούμε να λάβουμε διαφορετικές μορφές της ιδιότητας `Grades`. 

Εάν εκτελέσετε το παραπάνω κελί πολλές φορές, η μορφή μπορεί να είναι `3.7` ή `3.7 GPA`. 

Αυτό συμβαίνει επειδή το LLM παίρνει μη δομημένα δεδομένα με τη μορφή της γραπτής προτροπής και επιστρέφει επίσης μη δομημένα δεδομένα. Πρέπει να έχουμε μια δομημένη μορφή ώστε να ξέρουμε τι να περιμένουμε όταν αποθηκεύουμε ή χρησιμοποιούμε αυτά τα δεδομένα.

Χρησιμοποιώντας λειτουργική κλήση, μπορούμε να βεβαιωθούμε ότι λαμβάνουμε πίσω δομημένα δεδομένα. Όταν χρησιμοποιούμε λειτουργική κλήση, το LLM δεν καλεί ή εκτελεί πραγματικά καμία λειτουργία. Αντιθέτως, δημιουργούμε μια δομή για να ακολουθήσει το LLM στις απαντήσεις του. Στη συνέχεια, χρησιμοποιούμε αυτές τις δομημένες απαντήσεις για να ξέρουμε ποια λειτουργία να εκτελέσουμε στις εφαρμογές μας.  
 


![Διάγραμμα Ροής Κλήσης Συνάρτησης](../../../../translated_images/el/Function-Flow.083875364af4f4bb.webp)


Στη συνέχεια, μπορούμε να πάρουμε αυτό που επιστρέφεται από τη λειτουργία και να το στείλουμε πίσω στο LLM. Το LLM θα απαντήσει χρησιμοποιώντας φυσική γλώσσα για να απαντήσει στο ερώτημα του χρήστη. 


### Χρήσεις για τη χρήση κλήσεων συναρτήσεων 

**Κλήση Εξωτερικών Εργαλείων** 
Τα chatbots είναι εξαιρετικά στο να παρέχουν απαντήσεις σε ερωτήσεις χρηστών. Με τη χρήση κλήσεων συναρτήσεων, τα chatbots μπορούν να χρησιμοποιούν μηνύματα από χρήστες για να ολοκληρώσουν συγκεκριμένες εργασίες. Για παράδειγμα, ένας φοιτητής μπορεί να ζητήσει από το chatbot "Στείλε email στον καθηγητή μου λέγοντας ότι χρειάζομαι περισσότερη βοήθεια με αυτό το θέμα". Αυτό μπορεί να πραγματοποιήσει κλήση συνάρτησης `send_email(to: string, body: string)`


**Δημιουργία Ερωτημάτων API ή Βάσης Δεδομένων**
Οι χρήστες μπορούν να βρουν πληροφορίες χρησιμοποιώντας φυσική γλώσσα που μετατρέπεται σε διαμορφωμένο ερώτημα ή αίτημα API. Ένα παράδειγμα μπορεί να είναι ένας δάσκαλος που ζητά "Ποιοι είναι οι φοιτητές που ολοκλήρωσαν την τελευταία εργασία" το οποίο μπορεί να καλέσει μια συνάρτηση με όνομα `get_completed(student_name: string, assignment: int, current_status: string)`


**Δημιουργία Δομημένων Δεδομένων**
Οι χρήστες μπορούν να πάρουν ένα μπλοκ κειμένου ή CSV και να χρησιμοποιήσουν το LLM για να εξάγουν σημαντικές πληροφορίες από αυτό. Για παράδειγμα, ένας φοιτητής μπορεί να μετατρέψει ένα άρθρο της Wikipedia σχετικά με τις ειρηνευτικές συμφωνίες για να δημιουργήσει κάρτες flash AI. Αυτό μπορεί να γίνει με τη χρήση συνάρτησης που ονομάζεται `get_important_facts(agreement_name: string, date_signed: string, parties_involved: list)`


## 2. Δημιουργώντας την Πρώτη Κλήση Συνάρτησής σας 

Η διαδικασία δημιουργίας μιας κλήσης συνάρτησης περιλαμβάνει 3 βασικά βήματα: 
1. Κλήση του Chat Completions API με μια λίστα των συναρτήσεών σας και ένα μήνυμα χρήστη 
2. Ανάγνωση της απάντησης του μοντέλου για την εκτέλεση μιας ενέργειας π.χ. εκτέλεση συνάρτησης ή κλήση API 
3. Κάντε άλλη μια κλήση στο Chat Completions API με την απάντηση από τη συνάρτησή σας για να χρησιμοποιήσετε αυτές τις πληροφορίες για να δημιουργήσετε μια απάντηση προς τον χρήστη. 


![Ροή μιας Κλήσης Συνάρτησης](../../../../translated_images/el/LLM-Flow.3285ed8caf4796d7.webp)


### Στοιχεία μιας κλήσης συνάρτησης 

#### Είσοδος Χρήστη 

Το πρώτο βήμα είναι να δημιουργήσετε ένα μήνυμα χρήστη. Αυτό μπορεί να οριστεί δυναμικά λαμβάνοντας την τιμή από μια είσοδο κειμένου ή μπορείτε να αναθέσετε μια τιμή εδώ. Αν είναι η πρώτη σας φορά που εργάζεστε με το Chat Completions API, πρέπει να ορίσουμε το `role` και το `content` του μηνύματος. 

Το `role` μπορεί να είναι είτε `system` (δημιουργία κανόνων), `assistant` (το μοντέλο) ή `user` (ο τελικός χρήστης). Για την κλήση συναρτήσεων, θα το ορίσουμε ως `user` και με ένα παράδειγμα ερώτησης. 


In [ ]:
messages= [ {"role": "user", "content": "Find me a good course for a beginner student to learn Azure."} ]

### Δημιουργία συναρτήσεων. 

Στη συνέχεια θα ορίσουμε μια συνάρτηση και τις παραμέτρους αυτής της συνάρτησης. Θα χρησιμοποιήσουμε μόνο μία συνάρτηση εδώ που ονομάζεται `search_courses`, αλλά μπορείτε να δημιουργήσετε πολλαπλές συναρτήσεις.

**Σημαντικό** : Οι συναρτήσεις περιλαμβάνονται στο μήνυμα του συστήματος προς το LLM και θα συμπεριληφθούν στον αριθμό των διαθέσιμων tokens που έχετε στη διάθεσή σας. 


In [ ]:
# The Responses API uses a flat tool format: name/description/parameters at the top level
functions = [
   {
      "type":"function",
      "name":"search_courses",
      "description":"Retrieves courses from the search index based on the parameters provided",
      "parameters":{
         "type":"object",
         "properties":{
            "role":{
               "type":"string",
               "description":"The role of the learner (i.e. developer, data scientist, student, etc.)"
            },
            "product":{
               "type":"string",
               "description":"The product that the lesson is covering (i.e. Azure, Power BI, etc.)"
            },
            "level":{
               "type":"string",
               "description":"The level of experience the learner has prior to taking the course (i.e. beginner, intermediate, advanced)"
            }
         },
         "required":[
            "role"
         ]
      }
   }
]


**Ορισμοί** 

`name` - Το όνομα της συνάρτησης που θέλουμε να κληθεί. 

`description` - Αυτή είναι η περιγραφή του πώς λειτουργεί η συνάρτηση. Εδώ είναι σημαντικό να είμαστε συγκεκριμένοι και σαφείς 

`parameters` - Μια λίστα με τιμές και μορφή που θέλουμε το μοντέλο να παράγει στην απάντησή του 


`type` - Ο τύπος δεδομένων στον οποίο θα αποθηκευτούν οι ιδιότητες 

`properties` - Λίστα με τις συγκεκριμένες τιμές που το μοντέλο θα χρησιμοποιήσει για την απάντησή του 


`name` - το όνομα της ιδιότητας που το μοντέλο θα χρησιμοποιήσει στην μορφοποιημένη απάντησή του 

`type` - Ο τύπος δεδομένων αυτής της ιδιότητας 

`description` - Περιγραφή της συγκεκριμένης ιδιότητας 


**Προαιρετικό**

`required` - απαραίτητη ιδιότητα για να ολοκληρωθεί η κλήση της συνάρτησης 


### Κλήση της συνάρτησης 
Αφού ορίσουμε μια συνάρτηση, πρέπει τώρα να την συμπεριλάβουμε στην κλήση προς το Chat Completion API. Το κάνουμε αυτό προσθέτοντας `functions` στο αίτημα. Σε αυτή την περίπτωση, `functions=functions`. 

Υπάρχει επίσης η επιλογή να οριστεί το `function_call` σε `auto`. Αυτό σημαίνει ότι αφήνουμε το LLM να αποφασίσει ποια συνάρτηση θα κληθεί με βάση το μήνυμα του χρήστη, αντί να την καθορίσουμε εμείς.


In [ ]:
response = client.responses.create(model=deployment, 
                                        input=messages,
                                        tools=functions, 
                                        tool_choice="auto",
                                        store=False) 

print(response.output)


Τώρα ας δούμε την απάντηση και πώς είναι μορφοποιημένη: 

{
  "role": "assistant",
  "function_call": {
    "name": "search_courses",
    "arguments": "{\n  \"role\": \"student\",\n  \"product\": \"Azure\",\n  \"level\": \"beginner\"\n}"
  }
}

Μπορείτε να δείτε ότι καλείται το όνομα της συνάρτησης και από το μήνυμα του χρήστη, το LLM κατάφερε να βρει τα δεδομένα για να ταιριάξουν με τα επιχειρήματα της συνάρτησης. 


## 3.Ενσωμάτωση Κλήσεων Συναρτήσεων σε μια Εφαρμογή. 


Αφού δοκιμάσαμε την μορφοποιημένη απάντηση από το LLM, τώρα μπορούμε να την ενσωματώσουμε σε μια εφαρμογή. 

### Διαχείριση της ροής 

Για να το ενσωματώσουμε στην εφαρμογή μας, ας ακολουθήσουμε τα εξής βήματα: 

Πρώτα, ας κάνουμε την κλήση στις υπηρεσίες Open AI και αποθηκεύσουμε το μήνυμα σε μια μεταβλητή που ονομάζεται `response_message`. 


In [ ]:
# Extract the function call items from the response output
tool_calls = [item for item in response.output if item.type == "function_call"]


Τώρα θα ορίσουμε τη συνάρτηση που θα καλέσει το API του Microsoft Learn για να πάρει μια λίστα μαθημάτων: 


In [ ]:
import requests

def search_courses(role, product, level):
    url = "https://learn.microsoft.com/api/catalog/"
    params = {
        "role": role,
        "product": product,
        "level": level
    }
    response = requests.get(url, params=params)
    modules = response.json()["modules"]
    results = []
    for module in modules[:5]:
        title = module["title"]
        url = module["url"]
        results.append({"title": title, "url": url})
    return str(results)



Ως βέλτιστη πρακτική, στη συνέχεια θα δούμε αν το μοντέλο θέλει να καλέσει μια συνάρτηση. Μετά από αυτό, θα δημιουργήσουμε μια από τις διαθέσιμες συναρτήσεις και θα τη συνδυάσουμε με τη συνάρτηση που καλείται.
Στη συνέχεια θα πάρουμε τα ορίσματα της συνάρτησης και θα τα αντιστοιχίσουμε στα ορίσματα από το LLM.

Τέλος, θα προσαρτήσουμε το μήνυμα κλήσης της συνάρτησης και τις τιμές που επέστρεψε το μήνυμα `search_courses`. Αυτό δίνει στο LLM όλες τις πληροφορίες που χρειάζεται για
να απαντήσει στον χρήστη χρησιμοποιώντας φυσική γλώσσα.


In [ ]:
# Check if the model wants to call a function
if tool_calls:
    tool_call = tool_calls[0]
    print("Recommended Function call:")
    print(tool_call.name)
    print()

    # Call the function. 
    function_name = tool_call.name

    available_functions = {
            "search_courses": search_courses,
    }
    function_to_call = available_functions[function_name] 

    function_args = json.loads(tool_call.arguments)
    function_response = function_to_call(**function_args)

    print("Output of function call:")
    print(function_response)
    print(type(function_response))


    # Add the model's function call item(s) and our function result to the conversation.
    # The Responses API represents tool results as `function_call_output` items.
    messages.extend(response.output)  # adding the model's function_call item(s)
    messages.append( # adding function response to messages
        {
            "type": "function_call_output",
            "call_id": tool_call.call_id,
            "output": function_response,
        }
    )


Τώρα θα στείλουμε το ενημερωμένο μήνυμα στο LLM ώστε να λάβουμε μια απάντηση σε φυσική γλώσσα αντί για απάντηση μορφοποιημένη σε JSON API. 


In [ ]:
print("Messages in next request:")
print(messages)
print()

second_response = client.responses.create(
    input=messages,
    model=deployment,
    tools=functions,
    tool_choice="auto",
    temperature=0,
    store=False,
        )  # get a new response from the model where it can see the function response


print(second_response.output_text)


## Πρόκληση Κώδικα 

Σπουδαία δουλειά! Για να συνεχίσεις την εκμάθησή σου στο Azure Open AI Function Calling μπορείς να δημιουργήσεις: https://learn.microsoft.com/training/support/catalog-api-developer-reference?WT.mc_id=academic-105485-koreyst 
 - Περισσότερες παράμετροι της συνάρτησης που μπορεί να βοηθήσουν τους μαθητές να βρουν περισσότερα μαθήματα. Μπορείς να βρεις τις διαθέσιμες παραμέτρους API εδώ: 
 - Δημιουργία μιας άλλης κλήσης συνάρτησης που λαμβάνει περισσότερες πληροφορίες από τον μαθητή, όπως η μητρική του γλώσσα 
 - Δημιουργία χειρισμού σφαλμάτων όταν η κλήση συνάρτησης και/ή η κλήση API δεν επιστρέφει κατάλληλα μαθήματα 


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Αποποίηση ευθυνών**:
Αυτό το έγγραφο έχει μεταφραστεί χρησιμοποιώντας την υπηρεσία μετάφρασης με τεχνητή νοημοσύνη [Co-op Translator](https://github.com/Azure/co-op-translator). Ενώ επιδιώκουμε την ακρίβεια, παρακαλούμε να έχετε υπόψη ότι οι αυτοματοποιημένες μεταφράσεις ενδέχεται να περιέχουν λάθη ή ανακρίβειες. Το πρωτότυπο έγγραφο στη μητρική του γλώσσα πρέπει να θεωρείται η αυθεντική πηγή. Για κρίσιμες πληροφορίες, συνιστάται επαγγελματική ανθρώπινη μετάφραση. Δεν φέρουμε ευθύνη για τυχόν παρεξηγήσεις ή λανθασμένες ερμηνείες που προκύπτουν από τη χρήση αυτής της μετάφρασης.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
